# Matplotlib Mini-Project: Executive Credit Risk Report
### Credit Card Risk Analysis Project

This is a capstone-style mini project that pulls together everything from Phases 1-4 —
line plots, histograms, boxplots, bar/stacked bar charts, scatter plots, color mapping,
subplots, and correlation heatmaps — into one realistic deliverable: a portfolio risk
report you could actually hand to a credit risk committee.

It also introduces three small but genuinely useful techniques we hadn't covered yet:
- **Twin axes** (`ax.twinx()`) — two variables, two different scales, one shared x-axis
- **Axis tick formatting** — forcing an axis to display as a percentage
- **Log-scale axes** — taming skewed data directly on the axis, without transforming the column

**Format:** Each task has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. The tasks here are less scaffolded than the phase notebooks — each one
expects you to combine skills, the way a real analysis would.

Run the setup cell first.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import PercentFormatter
import seaborn as sns

np.random.seed(42)

n = 800

# Application dates spread across the last 24 months
start_date = pd.Timestamp('2024-07-01')
application_date = start_date + pd.to_timedelta(np.random.randint(0, 730, size=n), unit='D')

age = np.clip(np.random.normal(40, 12, n), 18, 80)
housing_status = np.random.choice(['Rent', 'Own', 'Mortgage'], size=n, p=[0.35, 0.25, 0.40])
loan_purpose = np.random.choice(
    ['Debt Consolidation', 'Credit Card', 'Home Improvement', 'Other'],
    size=n, p=[0.40, 0.30, 0.15, 0.15]
)
employment_type = np.random.choice(['Salaried', 'Self-Employed', 'Unemployed'],
                                    size=n, p=[0.6, 0.3, 0.1])

annual_income = np.random.lognormal(mean=10.8, sigma=0.4, size=n)
total_debt = annual_income * np.random.uniform(0.05, 0.55, size=n)
debt_to_income = (total_debt / annual_income) * 100

credit_score = np.clip(np.random.normal(680, 55, n), 300, 850)
credit_limit = np.clip(3000 + annual_income * 0.15 + np.random.normal(0, 2000, n), 500, None)

raw_risk = (debt_to_income / 100) * 0.6 + ((850 - credit_score) / 550) * 0.6
# Employment risk nudges probability up for unemployed applicants
employment_bump = np.where(employment_type == 'Unemployed', 0.15, 0)
default_probability = np.clip(raw_risk + employment_bump + np.random.normal(0, 0.08, n), 0.01, 0.95)
default = np.random.binomial(1, default_probability)

df = pd.DataFrame({
    'Application_Date': application_date,
    'Age': age,
    'Housing_Status': housing_status,
    'Loan_Purpose': loan_purpose,
    'Employment_Type': employment_type,
    'Annual_Income': annual_income,
    'Total_Debt': total_debt,
    'Debt_to_Income': debt_to_income,
    'Credit_Score': credit_score,
    'Credit_Limit': credit_limit,
    'Default_Probability': default_probability,
    'Default': default
})

print(df.shape)
df.head()


---
## Task 1: Monthly Portfolio Trend

**Q1.** Build a monthly summary: group `df` by `df['Application_Date'].dt.to_period('M')`,
and aggregate to get `Volume` (count of applications) and `Default_Rate` (mean of
`Default` * 100). Convert the resulting period index to strings (e.g. `'2024-07'`) so
it plots cleanly. Store the result as `monthly` with columns `Volume` and `Default_Rate`.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
monthly = df.groupby(df['Application_Date'].dt.to_period('M')).agg(
    Volume=('Default', 'size'),
    Default_Rate=('Default', 'mean')
)
monthly['Default_Rate'] = monthly['Default_Rate'] * 100
monthly.index = monthly.index.astype(str)

monthly.head()


**Q2.** Plot `monthly['Volume']` as a line chart against `monthly.index` (OO approach).
Rotate the x-tick labels 90 degrees (there are 24 of them, so they'll overlap otherwise),
and add a title and y-label.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly.index, monthly['Volume'], marker='o')
ax.set_xticks(range(len(monthly.index)))
ax.set_xticklabels(monthly.index, rotation=90)
ax.set_title("Monthly Loan Application Volume")
ax.set_ylabel("Applications")
plt.show()


**Q3 (New technique — Twin Axes).** A single chart often needs to show two metrics with
*different scales* against the same timeline — application volume (hundreds) and default
rate (a percentage) don't belong on the same y-axis. Plot `monthly['Volume']` as bars on
a primary Axes `ax1`. Then create a second Axes sharing the same x-axis with
`ax2 = ax1.twinx()`, and plot `monthly['Default_Rate']` as a red line on `ax2`. Give
`ax1` a y-label of `"Application Volume"` and `ax2` a y-label of `"Default Rate (%)"`.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(monthly.index, monthly['Volume'], color='steelblue', label='Volume')
ax1.set_ylabel("Application Volume")
ax1.set_xticks(range(len(monthly.index)))
ax1.set_xticklabels(monthly.index, rotation=90)

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly['Default_Rate'], color='red', marker='o', label='Default Rate')
ax2.set_ylabel("Default Rate (%)")

fig.suptitle("Monthly Volume vs. Default Rate")
plt.show()


**Q4 (New technique — Axis Formatting).** Repeat Q3's twin-axis chart, but this time
format `ax2`'s y-axis to display a `%` sign automatically using
`ax2.yaxis.set_major_formatter(PercentFormatter(xmax=100))` (`xmax=100` because
`Default_Rate` is already stored on a 0-100 scale, not 0-1). Also combine the legends
from both axes into a single legend by collecting each Axes' `get_legend_handles_labels()`
and passing the combined lists to one `ax1.legend()` call.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(monthly.index, monthly['Volume'], color='steelblue', label='Volume')
ax1.set_ylabel("Application Volume")
ax1.set_xticks(range(len(monthly.index)))
ax1.set_xticklabels(monthly.index, rotation=90)

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly['Default_Rate'], color='red', marker='o', label='Default Rate')
ax2.set_ylabel("Default Rate (%)")
ax2.yaxis.set_major_formatter(PercentFormatter(xmax=100))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

fig.suptitle("Monthly Volume vs. Default Rate", fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()


---
## Task 2: Credit Score Distribution & Risk Tiers

**Q5.** Plot a histogram of `Credit_Score` (30 bins), then draw two vertical lines with
`ax.axvline()`: one at the mean (solid, `color='red'`) and one at the median (dashed,
`color='black'`). Add a legend distinguishing the two.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['Credit_Score'], bins=30, edgecolor='black', color='steelblue')
ax.axvline(df['Credit_Score'].mean(), color='red', linestyle='-', label='Mean')
ax.axvline(df['Credit_Score'].median(), color='black', linestyle='--', label='Median')
ax.set_title("Credit Score Distribution")
ax.legend()
plt.show()


**Q6.** Derive a `Risk_Tier` column by binning `Credit_Score` with `pd.cut()` into three
groups using edges `[300, 600, 700, 850]` and labels `['High', 'Medium', 'Low']` (lower
score = higher risk). Then build a grouped, filled boxplot (`patch_artist=True`) of
`Debt_to_Income` for the three tiers, with x-tick labels and a title.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
df['Risk_Tier'] = pd.cut(df['Credit_Score'], bins=[300, 600, 700, 850],
                         labels=['High', 'Medium', 'Low'])

tier_order = ['Low', 'Medium', 'High']
dti_by_tier = [df.loc[df['Risk_Tier'] == t, 'Debt_to_Income'] for t in tier_order]

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(dti_by_tier, patch_artist=True,
                 flierprops=dict(marker='D', markerfacecolor='red', markersize=5))
for box, color in zip(bp['boxes'], ['seagreen', 'orange', 'tomato']):
    box.set_facecolor(color)

ax.set_xticklabels(tier_order)
ax.set_xlabel("Risk Tier")
ax.set_ylabel("Debt-to-Income Ratio (%)")
ax.set_title("Debt-to-Income by Risk Tier")
plt.show()


---
## Task 3: Categorical Comparisons

**Q7.** Compute default rate (%) per `Employment_Type`, sort descending, and plot as a
bar chart with `ax.bar_label()` showing each rate to 1 decimal place. Color the
`'Unemployed'` bar `'tomato'` and the other two `'steelblue'` (hint: build the color
list with a list comprehension based on the sorted index).

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
rate_by_employment = (df.groupby('Employment_Type')['Default'].mean() * 100).sort_values(ascending=False)
colors = ['tomato' if e == 'Unemployed' else 'steelblue' for e in rate_by_employment.index]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(rate_by_employment.index, rate_by_employment.values, color=colors)
ax.bar_label(bars, fmt='%.1f%%')
ax.set_title("Default Rate by Employment Type")
ax.set_ylabel("Default Rate (%)")
plt.show()


**Q8.** Build a percentage stacked bar chart showing loan outcome composition (paid vs.
defaulted, green/red) by `Loan_Purpose`, using `pd.crosstab()` converted to row-wise
percentages. Add in-segment text labels and a legend placed outside the plot.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
crosstab = pd.crosstab(df['Loan_Purpose'], df['Default'])
crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(crosstab_pct.index, crosstab_pct[0], label='Paid', color='seagreen')
ax.bar(crosstab_pct.index, crosstab_pct[1], bottom=crosstab_pct[0], label='Defaulted', color='tomato')

for purpose in crosstab_pct.index:
    paid_pct = crosstab_pct.loc[purpose, 0]
    default_pct = crosstab_pct.loc[purpose, 1]
    ax.text(purpose, paid_pct / 2, f"{paid_pct:.0f}%", ha='center', va='center', color='white')
    ax.text(purpose, paid_pct + default_pct / 2, f"{default_pct:.0f}%", ha='center', va='center', color='white')

ax.set_title("Loan Outcome by Purpose")
ax.set_ylabel("Percentage of Applicants")
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
plt.show()


---
## Task 4: Bivariate Relationships

**Q9.** Scatter `Annual_Income` (x) vs. `Credit_Limit` (y), colored by `Default_Probability`
with `cmap='Reds'`, `alpha=0.6`, and a colorbar. Fit and overlay a linear trend line
with `np.polyfit`.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
slope, intercept = np.polyfit(df['Annual_Income'], df['Credit_Limit'], 1)
x_line = np.linspace(df['Annual_Income'].min(), df['Annual_Income'].max(), 100)
y_line = slope * x_line + intercept

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(df['Annual_Income'], df['Credit_Limit'], c=df['Default_Probability'],
                      cmap='Reds', alpha=0.6)
ax.plot(x_line, y_line, color='black', linewidth=2, label='Trend')
fig.colorbar(scatter, ax=ax, label='Default Probability')
ax.set_xlabel("Annual Income")
ax.set_ylabel("Credit Limit")
ax.set_title("Income vs. Credit Limit, Colored by Default Risk")
ax.legend()
plt.show()


**Q10 (New technique — Log Scale).** Plot a histogram of `Annual_Income` using bin edges
from `np.logspace(np.log10(df['Annual_Income'].min()), np.log10(df['Annual_Income'].max()), 30)`,
then call `ax.set_xscale('log')`. This is a direct alternative to `np.log1p()`-transforming
the column first — the data stays in its original units, but the axis itself compresses
the long right tail, which is often the more interpretable choice for a report (readers
see real dollar amounts on the x-axis, not log-dollars).

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
log_bins = np.logspace(np.log10(df['Annual_Income'].min()), np.log10(df['Annual_Income'].max()), 30)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['Annual_Income'], bins=log_bins, edgecolor='black', color='seagreen')
ax.set_xscale('log')
ax.set_xlabel("Annual Income (log scale)")
ax.set_ylabel("Number of Applicants")
ax.set_title("Income Distribution — Log-Scaled Axis")
plt.show()


---
## Task 5: Feature Correlation

**Q11.** Build a correlation heatmap of `['Age', 'Annual_Income', 'Total_Debt', 'Debt_to_Income', 'Credit_Score', 'Credit_Limit', 'Default_Probability']`,
masking the upper triangle, with `annot=True`, `fmt='.2f'`, `cmap='coolwarm'`, `center=0`.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
numeric_features = ['Age', 'Annual_Income', 'Total_Debt', 'Debt_to_Income',
                    'Credit_Score', 'Credit_Limit', 'Default_Probability']
corr_matrix = df[numeric_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.show()


---
## Final Task: The Executive Risk Report

**Q12.** Assemble everything into a single 2x2 dashboard Figure (`figsize=(16, 11)`):
- **Top-left:** the twin-axis monthly volume + default rate chart from Q4 (volume bars +
  red default-rate line on a secondary axis, percentage-formatted)
- **Top-right:** the filled boxplot of `Debt_to_Income` by `Risk_Tier` from Q6
- **Bottom-left:** the sorted default-rate-by-employment bar chart from Q7
- **Bottom-right:** the color-mapped `Annual_Income` vs. `Credit_Limit` scatter from Q9,
  with its colorbar

Add one `fig.suptitle("Executive Credit Risk Report")` in bold, call `fig.tight_layout()`,
and save it to disk at 150 dpi with `fig.savefig('executive_risk_report.png', dpi=150, bbox_inches='tight')`.

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Top-left: twin-axis trend
ax1 = axes[0, 0]
ax1.bar(monthly.index, monthly['Volume'], color='steelblue', label='Volume')
ax1.set_ylabel("Application Volume")
ax1.set_xticks(range(len(monthly.index)))
ax1.set_xticklabels(monthly.index, rotation=90, fontsize=7)
ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly['Default_Rate'], color='red', marker='o', label='Default Rate')
ax2.set_ylabel("Default Rate (%)")
ax2.yaxis.set_major_formatter(PercentFormatter(xmax=100))
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
ax1.set_title("Monthly Volume vs. Default Rate")

# Top-right: DTI by risk tier
bp = axes[0, 1].boxplot(dti_by_tier, patch_artist=True,
                         flierprops=dict(marker='D', markerfacecolor='red', markersize=5))
for box, color in zip(bp['boxes'], ['seagreen', 'orange', 'tomato']):
    box.set_facecolor(color)
axes[0, 1].set_xticklabels(tier_order)
axes[0, 1].set_xlabel("Risk Tier")
axes[0, 1].set_ylabel("Debt-to-Income Ratio (%)")
axes[0, 1].set_title("Debt-to-Income by Risk Tier")

# Bottom-left: default rate by employment
bars = axes[1, 0].bar(rate_by_employment.index, rate_by_employment.values, color=colors)
axes[1, 0].bar_label(bars, fmt='%.1f%%')
axes[1, 0].set_ylabel("Default Rate (%)")
axes[1, 0].set_title("Default Rate by Employment Type")

# Bottom-right: income vs limit, colored by risk
scatter = axes[1, 1].scatter(df['Annual_Income'], df['Credit_Limit'],
                              c=df['Default_Probability'], cmap='Reds', alpha=0.6)
axes[1, 1].plot(x_line, y_line, color='black', linewidth=2)
fig.colorbar(scatter, ax=axes[1, 1], label='Default Probability')
axes[1, 1].set_xlabel("Annual Income")
axes[1, 1].set_ylabel("Credit Limit")
axes[1, 1].set_title("Income vs. Credit Limit")

fig.suptitle("Executive Credit Risk Report", fontsize=17, fontweight='bold')
fig.tight_layout()
fig.savefig('executive_risk_report.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved executive_risk_report.png")


---
## Mini-Project Complete

You just built a report that touches every Matplotlib topic from Phases 1-4, plus three
new techniques worth remembering:

- **Twin axes** (`ax.twinx()`) — whenever you need two differently-scaled metrics on one
  timeline (volume vs. rate is the classic finance example)
- **Tick formatting** (`PercentFormatter`, and more generally `FuncFormatter`) — let the
  axis handle display formatting instead of manually editing labels
- **Log-scale axes** (`set_xscale('log')`) — an alternative to transforming skewed
  financial data, keeping the original units visible to the reader

At this point you have the full Matplotlib + Seaborn-integration toolkit this project
needs. Natural next steps: apply this directly to your real dataset, or move on to a
dedicated Seaborn deep-dive for its higher-level statistical plotting shortcuts.
